# BuzzSpot Model B - YOLO26m 700×700 sliced train+val final-fit

This notebook trains the final-fit **Model B** slice-specialized YOLO26m model.

Changes from the earlier Model B notebook:

- starts from the new train+val full-frame Model A checkpoint: `yolo26m_32ep_rare_os_cm5_trainval_finalfit_last.pt`
- builds sliced training data from official **train + valid** annotated keyframes
- uses the same 700×700 SAHI-style slicing, overlap, empty-slice cap, rare-class oversampling, and sliced-inference/merge logic
- uses `last.pt` as the selected Model B checkpoint from the fixed 12-epoch run
- skips leaked validation and visual sanity-check sections because valid is now part of training

The A+B ensemble/merge still belongs in a separate notebook after Model B finishes.


## 1. Install


In [1]:
!pip install -q ultralytics tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.9 MB/s eta 0:00:00


## 2. Mount Drive and locate dataset files


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

zip_hits = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert zip_hits, "BuzzSet_challenge.zip not found under MyDrive"
ZIP_PATH = zip_hits[0]

tar_hits = glob.glob("/content/drive/MyDrive/**/19557529600-BuzzSet_challenge_testphase.tar", recursive=True)
if not tar_hits:
    tar_hits = glob.glob("/content/drive/MyDrive/**/*BuzzSet_challenge_testphase*.tar*", recursive=True)
assert tar_hits, "test-phase tar file not found under MyDrive"
TAR_PATH = tar_hits[0]

print("old train/valid zip:", ZIP_PATH)
print("new test-phase tar:", TAR_PATH)


Mounted at /content/drive
old train/valid zip: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/BuzzSet_challenge.zip
new test-phase tar: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/19557529600-BuzzSet_challenge_testphase.tar


## 3. Config


In [3]:
from pathlib import Path
import shutil

ENABLE_TRAINING = True
REBUILD_SLICED_DATASET = True

# New full-frame Model A final-fit checkpoint that scored 0.366 hidden test.
MODEL_A_TAG = "yolo26m_32ep_rare_os_cm5_trainval_finalfit"
MODEL_TAG = "yolo26m_sliced700_trainval_from_fullframeA_12ep"
RUN_NAME = "buzz_yolo26m_sliced700_trainval_from_fullframeA_12ep"

SLICE_SIZE = 700
OVERLAP = 0.20
STRIDE = int(SLICE_SIZE * (1.0 - OVERLAP))  # 560

MIN_VISIBLE_FRAC = 0.20
MIN_BOX_SIDE = 2.0

IMGSZ = 704
EPOCHS = 12
PATIENCE = 5
SAVE_PERIOD = 3
BATCH = 8  # if this OOMs on L4, use 4 or 6
SEED = 0

MOSAIC = 0.5
CLOSE_MOSAIC = 3

# Model B-only sliced inference settings.
# Keep low confidence for submission so the leaderboard evaluator can build the PR curve.
EVAL_CONF = 0.001
SUBMIT_CONF = 0.001
IOU = 0.45
MAX_DET_PER_SLICE = 100
AUGMENT = False

# Merge duplicate predictions from overlapping Model B slices.
# This is NOT Model A+B merging.
MERGE_METHOD = "nmm_ios"
MERGE_IOS = 0.50
MAX_DET_FINAL = 100
INFER_BATCH_SLICES = 8

LOCAL_DATA_DIR = Path("/content/buzz_modelB_sliced700_trainval")
SOURCE_DIR = LOCAL_DATA_DIR / "source_full"
SLICED_DIR = LOCAL_DATA_DIR / "sliced700"

# Selected Model B checkpoint for this final-fit notebook is last.pt, not best.pt.
LOCAL_WEIGHTS = Path("/content/modelB_trainval_last.pt")

PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
WEIGHTS_DIR = PROJECT_DIR / "weights"
DRIVE_RUNS_DIR = PROJECT_DIR / "runs_sliced700"
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

BASE_FULLFRAME_WEIGHTS = WEIGHTS_DIR / f"{MODEL_A_TAG}_last.pt"
DRIVE_SELECTED_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_last.pt"
DRIVE_BEST_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_best.pt"
DRIVE_LAST_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_last.pt"
DRIVE_RESULTS_CSV = WEIGHTS_DIR / f"{MODEL_TAG}_results.csv"

EVAL_CONF_TAG = f"evalconf{int(EVAL_CONF * 1000):03d}"
SUBMIT_CONF_TAG = f"conf{int(SUBMIT_CONF * 1000):03d}"
LOCAL_PRED_PATH = Path("/content/predictions.json")
LOCAL_ZIP_OUT = Path(f"/content/submission_{MODEL_TAG}_modelB_only_{SUBMIT_CONF_TAG}.zip")
DRIVE_PRED_PATH = SUBMISSIONS_DIR / f"predictions_{MODEL_TAG}_modelB_only_{SUBMIT_CONF_TAG}.json"
DRIVE_ZIP_OUT = SUBMISSIONS_DIR / f"submission_{MODEL_TAG}_modelB_only_{SUBMIT_CONF_TAG}.zip"

CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]

# Empty slices are useful hard background, but uncapped empty/background slices can dominate training.
# This cap applies only to the training TXT list; it does not delete files and it does not cap validation.
CAP_EMPTY_SLICES = True
EMPTY_TO_POS_RATIO = 3.0

CLASS_MULTIPLIER = {
    0: 1,  # bee
    1: 4,  # bumblebee
    2: 2,  # hoverfly
    3: 5,  # moth
}


def restore_selected_weights_to_local():
    """Restore the final-fit selected Model B checkpoint.

    For this train+val final-fit notebook, selected means last.pt from the fixed 12-epoch run.
    best.pt is still backed up for debugging, but should not be used for final-fit selection.
    """
    candidates = [
        DRIVE_SELECTED_WEIGHTS,
        DRIVE_LAST_WEIGHTS,
        DRIVE_RUNS_DIR / RUN_NAME / "weights" / "last.pt",
    ]

    for p in candidates:
        if p.exists():
            shutil.copy(p, LOCAL_WEIGHTS)
            print("Restored selected Model B last.pt from Drive:", p)
            print("Local weights:", LOCAL_WEIGHTS)
            return LOCAL_WEIGHTS

    if LOCAL_WEIGHTS.exists():
        print("WARNING: Drive Model B last.pt backup not found, but local selected weights exist:", LOCAL_WEIGHTS)
        return LOCAL_WEIGHTS

    raise FileNotFoundError(
        "No saved Model B last.pt found. Set ENABLE_TRAINING=True and run training first."
    )


# Backward-compatible alias so old cells do not silently use best.pt.
restore_best_weights_to_local = restore_selected_weights_to_local


def backup_training_outputs_to_drive():
    run_dir = DRIVE_RUNS_DIR / RUN_NAME
    best_path = run_dir / "weights" / "best.pt"
    last_path = run_dir / "weights" / "last.pt"
    results_csv = run_dir / "results.csv"

    assert last_path.exists(), f"last.pt not found at {last_path}"

    shutil.copy(last_path, LOCAL_WEIGHTS)
    shutil.copy(last_path, DRIVE_LAST_WEIGHTS)
    shutil.copy(last_path, DRIVE_SELECTED_WEIGHTS)
    print("Copied selected Model B last weights to local:", LOCAL_WEIGHTS)
    print("Backed up selected Model B last weights to Drive:", DRIVE_SELECTED_WEIGHTS)

    if best_path.exists():
        shutil.copy(best_path, DRIVE_BEST_WEIGHTS)
        print("Backed up Model B best weights to Drive for reference only:", DRIVE_BEST_WEIGHTS)

    if results_csv.exists():
        shutil.copy(results_csv, DRIVE_RESULTS_CSV)
        print("Backed up Model B results.csv to Drive:", DRIVE_RESULTS_CSV)


print("ENABLE_TRAINING:", ENABLE_TRAINING)
print("REBUILD_SLICED_DATASET:", REBUILD_SLICED_DATASET)
print("Model A base weights:", BASE_FULLFRAME_WEIGHTS)
print("Model B tag:", MODEL_TAG)
print("slice size:", SLICE_SIZE)
print("overlap:", OVERLAP)
print("stride:", STRIDE)
print("imgsz:", IMGSZ)
print("epochs:", EPOCHS)
print("batch:", BATCH)
print("seed:", SEED)
print("mosaic:", MOSAIC)
print("close_mosaic:", CLOSE_MOSAIC)
print("eval conf:", EVAL_CONF)
print("submit conf:", SUBMIT_CONF)
print("max det per slice:", MAX_DET_PER_SLICE)
print("cap empty slices:", CAP_EMPTY_SLICES)
print("empty-to-positive ratio:", EMPTY_TO_POS_RATIO)
print("selected Model B Drive weights:", DRIVE_SELECTED_WEIGHTS)
print("Model B-only Drive zip output:", DRIVE_ZIP_OUT)

assert BASE_FULLFRAME_WEIGHTS.exists(), (
    f"Base full-frame Model A train+val checkpoint missing: {BASE_FULLFRAME_WEIGHTS}\n"
    "Run/restore buzzspot_yolo26m_trainval_finalfit first, or copy the 0.366 test model to this path."
)


ENABLE_TRAINING: True
REBUILD_SLICED_DATASET: True
Model A base weights: /content/drive/MyDrive/BuzzSpot/weights/yolo26m_32ep_rare_os_cm5_trainval_finalfit_last.pt
Model B tag: yolo26m_sliced700_trainval_from_fullframeA_12ep
slice size: 700
overlap: 0.2
stride: 560
imgsz: 704
epochs: 12
batch: 8
seed: 0
mosaic: 0.5
close_mosaic: 3
eval conf: 0.001
submit conf: 0.001
max det per slice: 100
cap empty slices: True
empty-to-positive ratio: 3.0
selected Model B Drive weights: /content/drive/MyDrive/BuzzSpot/weights/yolo26m_sliced700_trainval_from_fullframeA_12ep_last.pt
Model B-only Drive zip output: /content/drive/MyDrive/BuzzSpot/submissions/submission_yolo26m_sliced700_trainval_from_fullframeA_12ep_modelB_only_conf001.zip


## 4. Build train+val 700×700 sliced YOLO dataset and extract test keyframes

In [4]:
import json, zipfile, tarfile, shutil, yaml
from pathlib import Path
from collections import defaultdict, Counter
from PIL import Image
from tqdm.auto import tqdm

if REBUILD_SLICED_DATASET and LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)

for d in [
    SOURCE_DIR / "images/train",
    SOURCE_DIR / "images/val",
    SOURCE_DIR / "images/test_testphase",
    LOCAL_DATA_DIR / "annotations",
    SLICED_DIR / "images/train",
    SLICED_DIR / "images/val",
    SLICED_DIR / "labels/train",
    SLICED_DIR / "labels/val",
]:
    d.mkdir(parents=True, exist_ok=True)

zf = zipfile.ZipFile(ZIP_PATH)
tf = tarfile.open(TAR_PATH, "r:*")

zip_names = set(zf.namelist())
tar_members = {m.name: m for m in tf.getmembers() if m.isfile()}
tar_names = set(tar_members.keys())


def find_zip(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}"):
        if cand in zip_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in zip_names if n.endswith(suffix)]
    return hits[0] if hits else None


def find_tar(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}", f"BuzzSet_challenge_testphase/{rel}"):
        if cand in tar_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in tar_names if n.endswith(suffix)]
    return hits[0] if hits else None


def load_tar_json(fname):
    p = find_tar(f"annotations/{fname}") or find_tar(fname)
    assert p is not None, f"{fname} not found in tar"
    with tf.extractfile(tar_members[p]) as f:
        obj = json.load(f)
    out = LOCAL_DATA_DIR / "annotations" / fname
    out.write_text(json.dumps(obj))
    print("loaded current annotation:", fname, "from", p)
    return obj


def flat_name(file_name):
    return file_name.replace("/", "__")


def annotations_by_image(coco):
    by_img = defaultdict(list)
    for ann in coco.get("annotations", []):
        by_img[int(ann["image_id"])].append(ann)
    return by_img


def extract_source_split(coco, zip_img_dir, out_split):
    """Extract only supervised/annotated keyframes for train/val.

    This deliberately matches the known-good Model A logic:
    - build by_img from annotations
    - iterate only image ids that appear in annotations
    - do NOT treat unannotated/context frames in coco["images"] as negative images

    Empty/background examples for Model B must come from 700x700 slices
    of annotated keyframes, not from unlabeled full context frames.
    """
    by_img = annotations_by_image(coco)
    images_by_id = {int(im["id"]): im for im in coco["images"]}

    # Critical: same selection rule as Model A.
    image_ids = sorted(by_img.keys())

    print(f"\n{out_split} COCO image records:", len(coco["images"]))
    print(f"{out_split} annotated image ids used:", len(image_ids))
    print(f"{out_split} skipped unannotated/context image records:", len(coco["images"]) - len(image_ids))

    missing_image_records = [iid for iid in image_ids if iid not in images_by_id]
    assert not missing_image_records, (
        f"{out_split}: {len(missing_image_records)} annotation image_ids not found in coco['images']; "
        f"first missing ids: {missing_image_records[:10]}"
    )

    records = []
    class_counts = Counter()
    box_count = 0

    for iid in tqdm(image_ids, desc=f"extract {out_split} annotated full images"):
        im = images_by_id[iid]

        src = find_zip(f"{zip_img_dir}/{im['file_name']}") or find_zip(im["file_name"])
        assert src is not None, f"missing image in old zip: {zip_img_dir}/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        img_dst = SOURCE_DIR / "images" / out_split / out_name

        if REBUILD_SLICED_DATASET or not img_dst.exists():
            with zf.open(src) as s, open(img_dst, "wb") as d:
                shutil.copyfileobj(s, d)

        anns = by_img[iid]
        assert len(anns) > 0, f"{out_split}: internal bug, selected image id {iid} has no annotations"

        for ann in anns:
            class_counts[int(ann["category_id"])] += 1
        box_count += len(anns)

        records.append({
            "id": iid,
            "file_name": im["file_name"],
            "flat_name": out_name,
            "path": str(img_dst),
            "width": int(im["width"]),
            "height": int(im["height"]),
            "annotations": anns,
        })

    print(out_split, "images:", len(records), "boxes:", box_count, "class counts:", dict(class_counts))
    print(out_split, "named counts:", {CLASS_NAMES[k-1]: class_counts[k] for k in range(1, 5)})

    expected_images = {"train": 5275, "val": 932}.get(out_split)
    expected_boxes = {"train": 10884, "val": 1116}.get(out_split)

    if expected_images is not None:
        assert len(records) == expected_images, (
            f"{out_split}: expected {expected_images} annotated images, got {len(records)}. "
            "Do not train until this matches Model A."
        )

    if expected_boxes is not None:
        assert box_count == expected_boxes, (
            f"{out_split}: expected {expected_boxes} boxes, got {box_count}. "
            "Do not train until this matches Model A/current annotations."
        )

    return records


def extract_test_keyframes(test_coco):
    keyframe_images = [
        im for im in test_coco["images"]
        if im.get("is_keyframe") in [True, 1, "true", "True"]
    ]

    records = []
    flat_to_id = {}

    for im in tqdm(keyframe_images, desc="extract test_testphase keyframes"):
        src = find_tar(f"test_testphase/{im['file_name']}") or find_tar(im["file_name"])
        assert src is not None, f"missing test image in tar: test_testphase/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        dst = SOURCE_DIR / "images" / "test_testphase" / out_name

        if REBUILD_SLICED_DATASET or not dst.exists():
            with tf.extractfile(tar_members[src]) as s, open(dst, "wb") as d:
                shutil.copyfileobj(s, d)

        flat_to_id[out_name] = int(im["id"])
        records.append({
            "id": int(im["id"]),
            "file_name": im["file_name"],
            "flat_name": out_name,
            "path": str(dst),
            "width": int(im.get("width", 1920)),
            "height": int(im.get("height", 1080)),
        })

    print("test_testphase keyframes:", len(records))
    return records, flat_to_id


def slice_starts(length, size=SLICE_SIZE, stride=STRIDE):
    if length <= size:
        return [0]
    starts = list(range(0, length - size + 1, stride))
    last = length - size
    if starts[-1] != last:
        starts.append(last)
    return sorted(set(starts))


def clip_ann_to_slice(ann, x0, y0, slice_size=SLICE_SIZE):
    x, y, w, h = map(float, ann["bbox"])
    x1, y1, x2, y2 = x, y, x + w, y + h

    sx1, sy1 = x0, y0
    sx2, sy2 = x0 + slice_size, y0 + slice_size

    ix1 = max(x1, sx1)
    iy1 = max(y1, sy1)
    ix2 = min(x2, sx2)
    iy2 = min(y2, sy2)

    iw = ix2 - ix1
    ih = iy2 - iy1

    if iw < MIN_BOX_SIDE or ih < MIN_BOX_SIDE:
        return None

    orig_area = max(1e-9, w * h)
    inter_area = iw * ih
    visible_frac = inter_area / orig_area

    if visible_frac < MIN_VISIBLE_FRAC:
        return None

    rx1 = ix1 - x0
    ry1 = iy1 - y0
    rx2 = ix2 - x0
    ry2 = iy2 - y0

    bw = rx2 - rx1
    bh = ry2 - ry1
    xc = (rx1 + bw / 2.0) / slice_size
    yc = (ry1 + bh / 2.0) / slice_size
    ww = bw / slice_size
    hh = bh / slice_size

    cls = int(ann["category_id"]) - 1

    if cls < 0 or cls >= len(CLASS_NAMES):
        return None

    return cls, xc, yc, ww, hh


def make_sliced_split(records, split):
    img_out_dir = SLICED_DIR / "images" / split
    lbl_out_dir = SLICED_DIR / "labels" / split
    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    slice_paths = []
    slice_class_counts = Counter()
    slice_image_class_counts = Counter()
    empty_slices = 0
    positive_slices = 0
    total_boxes = 0

    for rec in tqdm(records, desc=f"make {split} 700x700 slices"):
        img_path = Path(rec["path"])
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        xs = slice_starts(W)
        ys = slice_starts(H)

        base_stem = Path(rec["flat_name"]).stem

        for y0 in ys:
            for x0 in xs:
                slice_name = f"{base_stem}__x{x0}_y{y0}.jpg"
                slice_img_path = img_out_dir / slice_name
                slice_lbl_path = lbl_out_dir / f"{Path(slice_name).stem}.txt"

                if REBUILD_SLICED_DATASET or not slice_img_path.exists():
                    crop = img.crop((x0, y0, x0 + SLICE_SIZE, y0 + SLICE_SIZE))
                    crop.save(slice_img_path, quality=95)

                lines = []
                classes_in_slice = set()

                for ann in rec["annotations"]:
                    clipped = clip_ann_to_slice(ann, x0, y0)
                    if clipped is None:
                        continue

                    cls, xc, yc, ww, hh = clipped
                    lines.append(f"{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}")
                    classes_in_slice.add(cls)
                    slice_class_counts[cls] += 1
                    total_boxes += 1

                slice_lbl_path.write_text("\n".join(lines))

                if classes_in_slice:
                    positive_slices += 1
                    for c in classes_in_slice:
                        slice_image_class_counts[c] += 1
                else:
                    empty_slices += 1

                slice_paths.append(slice_img_path)

        img.close()

    print(f"\\n{split} sliced images:", len(slice_paths))
    print(f"{split} positive slices:", positive_slices)
    print(f"{split} empty slices:", empty_slices)
    print(f"{split} boxes after clipping:", total_boxes)
    print(f"{split} instance counts:", {CLASS_NAMES[k]: slice_class_counts[k] for k in range(4)})
    print(f"{split} slice-image counts:", {CLASS_NAMES[k]: slice_image_class_counts[k] for k in range(4)})

    return slice_paths


def label_classes(label_path):
    cls_set = set()
    if not Path(label_path).exists():
        return cls_set
    for line in Path(label_path).read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cls_set.add(int(float(parts[0])))
    return cls_set


def label_instance_counts(label_path):
    cnt = Counter()
    if not Path(label_path).exists():
        return cnt
    for line in Path(label_path).read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cnt[int(float(parts[0]))] += 1
    return cnt


def build_oversampled_train_txt(train_slice_paths):
    """Build the actual YOLO training list.

    Important: all slice image/label files remain on disk. This function only controls which
    image paths YOLO sees during training.

    Policy:
    - keep every positive slice
    - oversample positive slices using CLASS_MULTIPLIER
    - randomly sample empty/background slices up to EMPTY_TO_POS_RATIO × effective positive entries
    - do not cap validation; this function is train-only
    """
    import random

    rng = random.Random(SEED)

    positive_entries = []
    empty_paths = []

    unique_count = 0
    image_type_counts = Counter()
    original_instance_counts = Counter()
    effective_instance_counts = Counter()
    positive_unique_by_class = Counter()
    positive_effective_by_class = Counter()

    for img_path in train_slice_paths:
        img_path = Path(img_path)
        lbl_path = SLICED_DIR / "labels" / "train" / (img_path.stem + ".txt")
        cls_set = label_classes(lbl_path)
        inst_counts = label_instance_counts(lbl_path)
        unique_count += 1

        if cls_set:
            mult = max(CLASS_MULTIPLIER.get(c, 1) for c in cls_set)
            image_type_counts["positive_unique"] += 1
            for c in cls_set:
                positive_unique_by_class[c] += 1
                positive_effective_by_class[c] += mult

            original_instance_counts.update(inst_counts)

            for _ in range(mult):
                positive_entries.append(str(img_path.resolve()))
                effective_instance_counts.update(inst_counts)
        else:
            image_type_counts["empty_unique"] += 1
            empty_paths.append(str(img_path.resolve()))

    effective_positive_count = len(positive_entries)

    if CAP_EMPTY_SLICES:
        max_empty = int(effective_positive_count * EMPTY_TO_POS_RATIO)
        if len(empty_paths) > max_empty:
            selected_empty_paths = rng.sample(empty_paths, max_empty)
        else:
            selected_empty_paths = list(empty_paths)
    else:
        max_empty = len(empty_paths)
        selected_empty_paths = list(empty_paths)

    lines = positive_entries + selected_empty_paths
    rng.shuffle(lines)

    train_txt = SLICED_DIR / "train_sliced700_oversampled_empty_capped.txt"
    train_txt.write_text("\n".join(lines) + "\n")

    print("\ntrain unique slices:", unique_count)
    print("positive unique slices:", image_type_counts["positive_unique"])
    print("empty unique slices:", image_type_counts["empty_unique"])
    print("effective positive entries after oversampling:", effective_positive_count)
    print("empty cap enabled:", CAP_EMPTY_SLICES)
    print("EMPTY_TO_POS_RATIO:", EMPTY_TO_POS_RATIO)
    print("max empty allowed:", max_empty)
    print("selected empty entries:", len(selected_empty_paths))
    print("final train lines:", len(lines))
    print("final empty:positive ratio:", round(len(selected_empty_paths) / max(1, effective_positive_count), 3))
    print("positive unique slice counts by class:", {CLASS_NAMES[k]: positive_unique_by_class[k] for k in range(4)})
    print("positive effective slice entries by class:", {CLASS_NAMES[k]: positive_effective_by_class[k] for k in range(4)})
    print("original instance counts:", {CLASS_NAMES[k]: original_instance_counts[k] for k in range(4)})
    print("effective instance counts:", {CLASS_NAMES[k]: effective_instance_counts[k] for k in range(4)})
    print("train txt:", train_txt)

    if len(selected_empty_paths) == 0:
        print("WARNING: no empty/background slices selected. That is probably too aggressive.")
    if effective_positive_count == 0:
        raise RuntimeError("No positive training slices found. Check annotation remapping / slice generation.")

    return train_txt


train_coco = load_tar_json("train.json")
valid_coco = load_tar_json("valid.json")
test_coco = load_tar_json("test_testphase.json")

train_records = extract_source_split(train_coco, "train", "train")
valid_records = extract_source_split(valid_coco, "valid", "val")
test_records, flat_to_id = extract_test_keyframes(test_coco)


def with_slice_name_prefix(records, prefix):
    """Avoid train/valid slice filename collisions when both splits are added to train."""
    out = []
    for rec in records:
        r = dict(rec)
        r["flat_name"] = f"{prefix}__{rec['flat_name']}"
        out.append(r)
    return out


# Final-fit Model B training pool: official train + official valid annotated keyframes.
# A separate sliced val folder is still written only because the YOLO YAML expects a val path;
# validation is disabled during training and this notebook does not use val AP for selection.
trainval_records = with_slice_name_prefix(train_records, "train") + with_slice_name_prefix(valid_records, "valid")

print("\nfinal-fit Model B full-image training source:")
print("train annotated images:", len(train_records))
print("valid annotated images added:", len(valid_records))
print("train+valid annotated images:", len(trainval_records))
assert len(trainval_records) == 6207, f"Expected 6207 train+valid annotated images, got {len(trainval_records)}"

train_slice_paths = make_sliced_split(trainval_records, "train")
val_slice_paths = make_sliced_split(valid_records, "val")

TRAIN_TXT = build_oversampled_train_txt(train_slice_paths)

DATA_YAML = SLICED_DIR / "buzz_sliced700.yaml"
DATA_YAML.write_text(
    "\n".join([
        f"path: {SLICED_DIR}",
        f"train: {TRAIN_TXT}",
        "val: images/val",
        "names:",
        "  0: bee",
        "  1: bumblebee",
        "  2: hoverfly",
        "  3: moth",
        ""
    ])
)

print("\nDATA_YAML:")
print(DATA_YAML.read_text())
assert "\\n" not in DATA_YAML.read_text(), "YAML contains literal backslash-n characters"


loaded current annotation: train.json from BuzzSpot_testphase/annotations/train.json
loaded current annotation: valid.json from BuzzSpot_testphase/annotations/valid.json
loaded current annotation: test_testphase.json from BuzzSpot_testphase/annotations/test_testphase.json

train COCO image records: 31650
train annotated image ids used: 5275
train skipped unannotated/context image records: 26375


extract train annotated full images:   0%|          | 0/5275 [00:00<?, ?it/s]

train images: 5275 boxes: 10884 class counts: {1: 8677, 3: 1753, 2: 237, 4: 217}
train named counts: {'bee': 8677, 'bumblebee': 237, 'hoverfly': 1753, 'moth': 217}

val COCO image records: 5592
val annotated image ids used: 932
val skipped unannotated/context image records: 4660


extract val annotated full images:   0%|          | 0/932 [00:00<?, ?it/s]

val images: 932 boxes: 1116 class counts: {1: 934, 2: 30, 3: 95, 4: 57}
val named counts: {'bee': 934, 'bumblebee': 30, 'hoverfly': 95, 'moth': 57}


...

test_testphase keyframes: 4763

final-fit Model B full-image training source:
train annotated images: 5275
valid annotated images added: 932
train+valid annotated images: 6207


make train 700x700 slices:   0%|          | 0/6207 [00:00<?, ?it/s]

\ntrain sliced images: 49656
train positive slices: 19592
train empty slices: 30064
train boxes after clipping: 24232
train instance counts: {'bee': 19285, 'bumblebee': 524, 'hoverfly': 3876, 'moth': 547}
train slice-image counts: {'bee': 16104, 'bumblebee': 520, 'hoverfly': 3653, 'moth': 539}


make val 700x700 slices:   0%|          | 0/932 [00:00<?, ?it/s]

\nval sliced images: 7456
val positive slices: 1913
val empty slices: 5543
val boxes after clipping: 2015
val instance counts: {'bee': 1529, 'bumblebee': 69, 'hoverfly': 260, 'moth': 157}
val slice-image counts: {'bee': 1474, 'bumblebee': 67, 'hoverfly': 260, 'moth': 154}

train unique slices: 49656
positive unique slices: 19592
empty unique slices: 30064
effective positive entries after oversampling: 26900
empty cap enabled: True
EMPTY_TO_POS_RATIO: 3.0
max empty allowed: 80700
selected empty entries: 30064
final train lines: 56964
final empty:positive ratio: 1.118
positive unique slice counts by class: {'bee': 16104, 'bumblebee': 520, 'hoverfly': 3653, 'moth': 539}
positive effective slice entries by class: {'bee': 17836, 'bumblebee': 2087, 'hoverfly': 7388, 'moth': 2695}
original instance counts: {'bee': 19285, 'bumblebee': 524, 'hoverfly': 3876, 'moth': 547}
effective instance counts: {'bee': 21365, 'bumblebee': 2103, 'hoverfly': 7834, 'moth': 2735}
train txt: /content/buzz_modelB_

In [5]:
# Verify/rewrite YAML with real newlines, not literal "\n"

DATA_YAML = SLICED_DIR / "buzz_sliced700.yaml"

yaml_text = "\n".join([
    f"path: {SLICED_DIR}",
    f"train: {TRAIN_TXT}",
    "val: images/val",
    "names:",
    "  0: bee",
    "  1: bumblebee",
    "  2: hoverfly",
    "  3: moth",
    ""
])

DATA_YAML.write_text(yaml_text)

print("RAW REPR:")
print(repr(DATA_YAML.read_text()))

print("\nNORMAL PRINT:")
print(DATA_YAML.read_text())

assert "\\n" not in DATA_YAML.read_text(), "YAML still contains literal backslash-n characters"
assert "\ntrain:" in DATA_YAML.read_text(), "YAML does not contain real newline before train"
assert DATA_YAML.exists()


RAW REPR:
'path: /content/buzz_modelB_sliced700_trainval/sliced700\ntrain: /content/buzz_modelB_sliced700_trainval/sliced700/train_sliced700_oversampled_empty_capped.txt\nval: images/val\nnames:\n  0: bee\n  1: bumblebee\n  2: hoverfly\n  3: moth\n'

NORMAL PRINT:
path: /content/buzz_modelB_sliced700_trainval/sliced700
train: /content/buzz_modelB_sliced700_trainval/sliced700/train_sliced700_oversampled_empty_capped.txt
val: images/val
names:
  0: bee
  1: bumblebee
  2: hoverfly
  3: moth



## 5. Train final-fit Model B from train+val Model A checkpoint on train+val sliced dataset


In [6]:
from ultralytics import YOLO
import torch, gc

if ENABLE_TRAINING:
    print("ENABLE_TRAINING=True -> training Model B will run.")
    print("Starting from Model A checkpoint:", BASE_FULLFRAME_WEIGHTS)

    model = YOLO(str(BASE_FULLFRAME_WEIGHTS))

    train_kwargs = dict(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        save_period=SAVE_PERIOD,
        project=str(DRIVE_RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        seed=SEED,
        deterministic=True,
        cache="disk",
        workers=4,
        device=0,
        optimizer="auto",
        mosaic=MOSAIC,
        close_mosaic=CLOSE_MOSAIC,
        val=False,
    )

    print("training args:", train_kwargs)
    model.train(**train_kwargs)

    backup_training_outputs_to_drive()

    gc.collect()
    torch.cuda.empty_cache()

else:
    print("ENABLE_TRAINING=False -> skipping Model B training.")
    restore_selected_weights_to_local()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ENABLE_TRAINING=True -> training Model B will run.
Starting from Model A checkpoint: /content/drive/MyDrive/BuzzSpot/weights/yolo26m_32ep_rare_os_cm5_trainval_finalfit_last.pt
training args: {'data': '/content/buzz_modelB_sliced700_trainval/sliced700/buzz_sliced700.yaml', 'epochs': 12, 'imgsz': 704, 'batch': 8, 'patience': 5, 'save_period': 3, 'project': '/content/drive/MyDrive/BuzzSpot/runs_sliced700', 'name': 'buzz_yolo26m_sliced700_trainval_from_fullframeA_12ep', 'exist_ok': True, 'seed': 0, 'deterministic': True, 'cache': 'disk', 'workers': 4, 'device': 0, 'optimizer': 'auto', 'mosaic': 0.5, 'close_mosaic': 3, 'val': False}
Ultralytics 8.4.86 🚀 Python-3.12.13 torch-2.11.0+cu12

## 6. Define Model B sliced inference utilities

In [7]:
import json, gc, torch
from pathlib import Path
from collections import Counter
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

restore_selected_weights_to_local()
model = YOLO(str(LOCAL_WEIGHTS))



def xyxy_area(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def intersection_area(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)


def ios_xyxy(a, b):
    inter = intersection_area(a, b)
    denom = min(xyxy_area(a), xyxy_area(b))
    if denom <= 0:
        return 0.0
    return inter / denom


def weighted_merge_cluster(cluster):
    scores = np.array([p["score"] for p in cluster], dtype=np.float64)
    boxes = np.array([p["bbox_xyxy"] for p in cluster], dtype=np.float64)

    if scores.sum() <= 0:
        weights = np.ones_like(scores) / len(scores)
    else:
        weights = scores / scores.sum()

    merged_box = (boxes * weights[:, None]).sum(axis=0).tolist()
    merged_score = float(scores.max())

    p0 = cluster[0].copy()
    p0["bbox_xyxy"] = merged_box
    p0["score"] = merged_score
    return p0


def merge_predictions_greedy_nmm_ios(raw_preds, ios_thr=MERGE_IOS, max_det=MAX_DET_FINAL):
    final = []

    for category_id in sorted(set(p["category_id"] for p in raw_preds)):
        cls_preds = [p for p in raw_preds if p["category_id"] == category_id]
        cls_preds = sorted(cls_preds, key=lambda p: p["score"], reverse=True)

        while cls_preds:
            top = cls_preds[0]
            cluster = [top]
            rest = []

            for p in cls_preds[1:]:
                if ios_xyxy(top["bbox_xyxy"], p["bbox_xyxy"]) >= ios_thr:
                    cluster.append(p)
                else:
                    rest.append(p)

            final.append(weighted_merge_cluster(cluster))
            cls_preds = rest

    final = sorted(final, key=lambda p: p["score"], reverse=True)[:max_det]
    return final


def clip_xyxy(x1, y1, x2, y2, W, H):
    x1 = max(0.0, min(float(x1), W - 1))
    y1 = max(0.0, min(float(y1), H - 1))
    x2 = max(0.0, min(float(x2), W))
    y2 = max(0.0, min(float(y2), H))
    return x1, y1, x2, y2


def predict_image_model_b_sliced(img_path, image_id, W=None, H=None, infer_conf=EVAL_CONF):
    img_path = Path(img_path)
    img = Image.open(img_path).convert("RGB")

    if W is None or H is None:
        W, H = img.size

    raw_preds = []
    jobs = []
    batch_imgs = []

    xs = slice_starts(W)
    ys = slice_starts(H)

    def flush_batch(batch_imgs, jobs):
        batch_raw = []
        if not batch_imgs:
            return batch_raw

        results = model.predict(
            source=batch_imgs,
            imgsz=IMGSZ,
            conf=infer_conf,
            iou=IOU,
            max_det=MAX_DET_PER_SLICE,
            batch=len(batch_imgs),
            augment=AUGMENT,
            agnostic_nms=False,
            verbose=False,
        )

        for r, (sx, sy) in zip(results, jobs):
            if r.boxes is None or len(r.boxes) == 0:
                continue

            for b in r.boxes:
                x1, y1, x2, y2 = b.xyxy[0].tolist()
                x1, y1, x2, y2 = x1 + sx, y1 + sy, x2 + sx, y2 + sy
                x1, y1, x2, y2 = clip_xyxy(x1, y1, x2, y2, W, H)

                if x2 - x1 >= 1 and y2 - y1 >= 1:
                    batch_raw.append({
                        "image_id": int(image_id),
                        "category_id": int(b.cls[0]) + 1,
                        "bbox_xyxy": [x1, y1, x2, y2],
                        "score": float(b.conf[0]),
                    })

        return batch_raw

    for y0 in ys:
        for x0 in xs:
            crop = img.crop((x0, y0, x0 + SLICE_SIZE, y0 + SLICE_SIZE))

            # PIL gives RGB. Ultralytics NumPy prediction expects OpenCV-style BGR.
            crop_bgr = np.ascontiguousarray(np.array(crop)[:, :, ::-1])

            batch_imgs.append(crop_bgr)
            jobs.append((x0, y0))

            if len(batch_imgs) == INFER_BATCH_SLICES:
                raw_preds.extend(flush_batch(batch_imgs, jobs))
                batch_imgs = []
                jobs = []

    if batch_imgs:
        raw_preds.extend(flush_batch(batch_imgs, jobs))

    img.close()

    merged = merge_predictions_greedy_nmm_ios(raw_preds, ios_thr=MERGE_IOS, max_det=MAX_DET_FINAL)

    coco_preds = []
    for p in merged:
        x1, y1, x2, y2 = p["bbox_xyxy"]
        coco_preds.append({
            "image_id": int(p["image_id"]),
            "category_id": int(p["category_id"]),
            "bbox": [round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)],
            "score": round(float(p["score"]), 5),
        })

    return coco_preds, len(raw_preds), len(merged)

print("Selected Model B weights loaded for sliced inference:", LOCAL_WEIGHTS)
print("Sliced inference utilities ready.")


Restored selected Model B last.pt from Drive: /content/drive/MyDrive/BuzzSpot/weights/yolo26m_sliced700_trainval_from_fullframeA_12ep_last.pt
Local weights: /content/modelB_trainval_last.pt
Selected Model B weights loaded for sliced inference: /content/modelB_trainval_last.pt
Sliced inference utilities ready.


## 7. Model B-only sliced inference on test_testphase and create zip

In [8]:
import json, zipfile, gc, torch, shutil
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter

# Assumes section 6 has loaded `model` from the selected Model B last.pt.
assert 'model' in globals(), 'Run section 6 first so Model B weights and sliced inference utilities are loaded.'

gc.collect()
torch.cuda.empty_cache()

print("\nModel B-only test sliced inference config:")
print("SUBMIT_CONF:", SUBMIT_CONF)
print("IOU:", IOU)
print("MAX_DET_PER_SLICE:", MAX_DET_PER_SLICE)
print("MERGE_IOS:", MERGE_IOS)
print("MAX_DET_FINAL:", MAX_DET_FINAL)

test_preds = []
raw_total = 0
merged_total = 0

for i, rec in enumerate(tqdm(test_records, desc="Model B-only sliced inference on test")):
    preds_i, raw_i, merged_i = predict_image_model_b_sliced(
        rec["path"],
        image_id=rec["id"],
        W=rec["width"],
        H=rec["height"],
        infer_conf=SUBMIT_CONF,
    )

    test_preds.extend(preds_i)
    raw_total += raw_i
    merged_total += merged_i

    if i % 100 == 0:
        print(i, "/", len(test_records), "raw:", raw_total, "merged:", len(test_preds))
        gc.collect()
        torch.cuda.empty_cache()

with open(LOCAL_PRED_PATH, "w") as f:
    json.dump(test_preds, f)

with zipfile.ZipFile(LOCAL_ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(LOCAL_PRED_PATH, arcname="predictions.json")

shutil.copy(LOCAL_PRED_PATH, DRIVE_PRED_PATH)
shutil.copy(LOCAL_ZIP_OUT, DRIVE_ZIP_OUT)

print("\\nraw test preds before merge:", raw_total)
print("detections after merge:", len(test_preds))
print("preds/image after merge:", len(test_preds) / max(1, len(test_records)))
print("class counts:", dict(Counter(int(p["category_id"]) for p in test_preds)))
print("local predictions:", LOCAL_PRED_PATH)
print("local zip:", LOCAL_ZIP_OUT)
print("Drive predictions:", DRIVE_PRED_PATH)
print("Drive zip:", DRIVE_ZIP_OUT)



Model B-only test sliced inference config:
SUBMIT_CONF: 0.001
IOU: 0.45
MAX_DET_PER_SLICE: 100
MERGE_IOS: 0.5
MAX_DET_FINAL: 100


Model B-only sliced inference on test:   0%|          | 0/4763 [00:00<?, ?it/s]

0 / 4763 raw: 22 merged: 12
100 / 4763 raw: 3453 merged: 841
200 / 4763 raw: 4314 merged: 1066
300 / 4763 raw: 5270 merged: 1352
400 / 4763 raw: 6042 merged: 1610
500 / 4763 raw: 6802 merged: 1886
600 / 4763 raw: 7507 merged: 2168
700 / 4763 raw: 8398 merged: 2525
800 / 4763 raw: 9156 merged: 2793
900 / 4763 raw: 9946 merged: 3084
1000 / 4763 raw: 11650 merged: 3684
1100 / 4763 raw: 13434 merged: 4243
1200 / 4763 raw: 14807 merged: 4691
1300 / 4763 raw: 16324 merged: 5088
1400 / 4763 raw: 18859 merged: 5759
1500 / 4763 raw: 20974 merged: 6291
1600 / 4763 raw: 22993 merged: 6833
1700 / 4763 raw: 24633 merged: 7391
1800 / 4763 raw: 26330 merged: 7871
1900 / 4763 raw: 28611 merged: 8520
2000 / 4763 raw: 31852 merged: 9320
2100 / 4763 raw: 34441 merged: 10006
2200 / 4763 raw: 36399 merged: 10706
2300 / 4763 raw: 39661 merged: 11524
2400 / 4763 raw: 42766 merged: 12372
2500 / 4763 raw: 48572 merged: 13664
2600 / 4763 raw: 56336 merged: 15932
2700 / 4763 raw: 58206 merged: 16353
2800 / 4763 

## 8. Validate Model B-only submission zip before upload

In [9]:
import json, zipfile
from pathlib import Path
from collections import Counter

p = json.load(open(LOCAL_PRED_PATH))
ids = {int(d["image_id"]) for d in p}
keyframe_ids = {int(rec["id"]) for rec in test_records}

print("detections:", len(p))
print("distinct predicted images:", len(ids))
print("total keyframe images:", len(keyframe_ids))
print("predictions outside keyframes:", len(ids - keyframe_ids))
print("keyframes with no detections:", len(keyframe_ids - ids))
print("categories:", sorted({int(d["category_id"]) for d in p}))
print("class counts:", dict(Counter(int(d["category_id"]) for d in p)))

print("degenerate:", sum(1 for d in p if d["bbox"][2] < 1 or d["bbox"][3] < 1))
print("out of bounds:", sum(
    1 for d in p
    if d["bbox"][0] < -0.5
    or d["bbox"][1] < -0.5
    or d["bbox"][0] + d["bbox"][2] > 1920.5
    or d["bbox"][1] + d["bbox"][3] > 1080.5
))

with zipfile.ZipFile(LOCAL_ZIP_OUT) as z:
    contents = z.namelist()
    print("zip contents:", contents)

assert len(ids - keyframe_ids) == 0, "Submission has predictions outside keyframes"
assert sorted({int(d["category_id"]) for d in p}) and set(int(d["category_id"]) for d in p).issubset({1,2,3,4}), "Bad category ids"
assert all(d["bbox"][2] >= 1 and d["bbox"][3] >= 1 for d in p), "Degenerate boxes exist"
assert contents == ["predictions.json"], "Zip must contain exactly predictions.json"

print("\nUpload this Model B-only zip if you want to test sliced model alone:")
print(DRIVE_ZIP_OUT)


detections: 41173
distinct predicted images: 4739
total keyframe images: 4763
predictions outside keyframes: 0
keyframes with no detections: 24
categories: [1, 2, 3, 4]
class counts: {1: 22383, 3: 15307, 2: 1409, 4: 2074}
degenerate: 0
out of bounds: 0
zip contents: ['predictions.json']

Upload this Model B-only zip if you want to test sliced model alone:
/content/drive/MyDrive/BuzzSpot/submissions/submission_yolo26m_sliced700_trainval_from_fullframeA_12ep_modelB_only_conf001.zip
